[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GarretOS/dq-ai-engineer-python/blob/main/projects/03-dynamic-ai-chatbot/dynamic_ai_chatbot.ipynb)

# 🤖 Dynamic AI Chatbot

This project demonstrates how to build a conversational chatbot powered by Google's Gemini API. It showcases API integration, conversation context management, token counting with tiktoken, persona switching, and building an interactive command-line interface. Run the final cells to chat with the AI in Google Colab.

## 🔑 Setup: Provide Your Gemini API Key

To run this notebook, you'll need a Gemini API key. Get one free at https://aistudio.google.com/apikey

In Google Colab, the cell below will prompt you to enter your API key securely (it won't be saved or displayed).

In [ ]:
import os
from google.colab import userdata
from getpass import getpass

try:
    # In Colab, securely retrieve the API key from secrets
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print("✓ Gemini API key loaded from Colab secrets")
except userdata.SecretNotFoundError:
    # If not set in Colab secrets, prompt for manual entry
    api_key = getpass('Enter your Gemini API key: ')
    os.environ['GEMINI_API_KEY'] = api_key
    print("✓ API key configured")

## 📦 Install Dependencies

Install the required packages for the chatbot.

In [ ]:
import subprocess
import sys

packages = ['openai', 'tiktoken']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ Dependencies installed")

## 🗣️ ConversationManager Class

The `ConversationManager` class handles all interactions with the Gemini API.
It manages conversation history, token counting, persona switching, and API communication.

In [ ]:
import tiktoken
from openai import OpenAI
import datetime
import json


class ConversationManager:
    """
    Manages communication between the chatbot and the AI model.
    """

    def __init__(
        self,
        model="gemini-3.5-flash",
        temperature=0.7,
        max_tokens=500,
        token_budget=4000,
        system_message="You are a thoughtful assistant who explains things clearly and step-by-step.",
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        history_file=None,
    ):
        # Get the API key securely from the environment variable
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("GEMINI_API_KEY environment variable is not set.")
        
        # Store model and response settings
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.token_budget = token_budget

        # Store the conversation history file location
        self.history_file = history_file
        if self.history_file is None:
            self.history_file = f"conversation_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

        # Create the OpenAI client using Gemini's OpenAI-compatible API
        self.client = OpenAI(
            api_key=api_key,
            base_url=base_url,
        )

        # Define the chatbot's system message
        self.system_message = system_message

        # Store predefined personas for easy switching
        self.system_messages = {
            "sassy_assistant": "You are a sassy assistant who is fed up with answering questions.",
            "angry_assistant": "You are an angry assistant who likes yelling in ALL CAPS.",
            "thoughtful_assistant": (
                "You are a thoughtful assistant who is always ready to dig deeper. "
                "Ask clarifying questions to ensure understanding and approach problems "
                "with a step-by-step methodology."
            ),
            "custom": system_message,
        }

        # Initialize conversation history with system message
        self.conversation_history = [
            {"role": "system", "content": self.system_message}
        ]

        # Load saved conversation history if it exists
        self.load_conversation_history()

    def load_conversation_history(self):
        """Load conversation history from the history file."""
        try:
            with open(self.history_file, "r") as file:
                self.conversation_history = json.load(file)
        except FileNotFoundError:
            print("No previous conversation history found. Starting new conversation.")
            self.conversation_history = [{"role": "system", "content": self.system_message}]
        except json.JSONDecodeError:
            print("Conversation history file is invalid. Starting new conversation.")
            self.conversation_history = [{"role": "system", "content": self.system_message}]
        except Exception as error:
            print(f"Could not load conversation history: {error}")
            self.conversation_history = [{"role": "system", "content": self.system_message}]

    def save_conversation_history(self):
        """Save the current conversation history to the history file."""
        try:
            with open(self.history_file, "w") as file:
                json.dump(self.conversation_history, file, indent=4)
        except Exception as error:
            print(f"Could not save conversation history: {error}")

    def set_persona(self, persona):
        """Switch the chatbot to one of the predefined personas."""
        try:
            if persona not in self.system_messages:
                raise ValueError(f"Unknown persona: {persona}")
            self.system_message = self.system_messages[persona]
            self.update_system_message_in_history()
        except Exception as error:
            print(f"Could not change persona: {error}")
   
    def set_custom_system_message(self, message):
        """Set a custom system message for the chatbot."""
        if not message.strip():
            raise ValueError("Custom system message cannot be empty.")
        self.system_messages["custom"] = message
        self.system_message = message
        self.update_system_message_in_history()

    def update_system_message_in_history(self):
        """Update the system message stored in the conversation history."""
        self.conversation_history[0] = {
            "role": "system",
            "content": self.system_message,
        }

    def count_tokens(self, text):
        """Count the number of tokens in a piece of text using cl100k_base encoding."""
        encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(text))

    def total_tokens_used(self):
        """Calculate the total number of tokens in the conversation history."""
        total = 0
        for message in self.conversation_history:
            total += self.count_tokens(message["content"])
        return total

    def enforce_token_budget(self):
        """Remove oldest messages until conversation fits within token budget."""
        try:
            while self.total_tokens_used() > self.token_budget:
                # Stop if only the system message remains
                if len(self.conversation_history) <= 1:
                    break
                # Remove the oldest conversation message
                self.conversation_history.pop(1)
        except Exception as error:
            print(f"Could not enforce token budget: {error}")

    def chat_completion(
        self, prompt, temperature=None, max_tokens=None, model=None
    ):
        """Send a user's prompt to the AI model and return its response."""
        # Use provided values or fall back to defaults
        temperature = temperature if temperature is not None else self.temperature
        max_tokens = max_tokens if max_tokens is not None else self.max_tokens
        model = model if model is not None else self.model
        
        # Add user message to conversation history
        self.conversation_history.append({"role": "user", "content": prompt})

        # Calculate and display token usage
        total_tokens = self.total_tokens_used()
        print(f"Total tokens used: {total_tokens}")

        # Enforce token budget to manage costs
        self.enforce_token_budget()

        try:
            # Send to Gemini API
            response = self.client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=self.conversation_history,
            )

            # Extract assistant's response
            assistant_message = response.choices[0].message.content

            # Add response to conversation history
            self.conversation_history.append(
                {"role": "assistant", "content": assistant_message}
            )
   
            # Save updated history
            self.save_conversation_history()
           
            return assistant_message

        except Exception as error:
            print(f"Error communicating with the AI model: {error}")
            return None

## 💬 Interactive Demo

Chat with the AI chatbot below. Try switching personas, asking follow-up questions,
and observe how the chatbot remembers your conversation history.

In [ ]:
# Create a chatbot instance
chatbot = ConversationManager(
    system_message="You are a thoughtful assistant who explains things clearly and step-by-step.",
    token_budget=2000,
)

print("🤖 Dynamic AI Chatbot")
print("=" * 50)
print("Commands: 'help' for options, 'quit' to exit")
print()

In [ ]:
# Interactive chat loop
while True:
    user_input = input("You: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() == "quit":
        print("Goodbye!")
        break
    
    if user_input.lower() == "help":
        print("\nAvailable commands:")
        print("  persona <name>  - Switch persona (sassy_assistant, angry_assistant, thoughtful_assistant)")
        print("  custom <message> - Set a custom system message")
        print("  clear           - Start a new conversation")
        print("  quit            - Exit the chatbot")
        print()
        continue
    
    if user_input.lower().startswith("persona "):
        persona_name = user_input[8:].strip()
        chatbot.set_persona(persona_name)
        print(f"Persona changed to: {persona_name}\n")
        continue
    
    if user_input.lower().startswith("custom "):
        custom_message = user_input[7:].strip()
        chatbot.set_custom_system_message(custom_message)
        print(f"Custom system message set.\n")
        continue
    
    if user_input.lower() == "clear":
        chatbot.conversation_history = [
            {"role": "system", "content": chatbot.system_message}
        ]
        print("Conversation cleared.\n")
        continue
    
    # Send message to chatbot
    response = chatbot.chat_completion(user_input)
    if response:
        print(f"Assistant: {response}\n")